### TRAIN
Entrenaremos los modelos para las n,s y m versiones de YOLO. Luego, guardar las gráficas  sobre el proceso de
entrenamiento con las que analizar su comportamiento y así establecer hipótesis como si
se produce sobreajuste o subajuste.

In [ ]:
# Importar librerias necesarias
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import glob

### Configuración

In [ ]:
# Ruta absoluta al dataset creado en el paso anterior
# YOLO necesita saber donde esta la carpeta 'train' y 'test'
dataset_path = os.path.abspath("dataset_isic")

# Modelos solicitados en el enunciado: nano, small, medium
modelos = ["yolov8n-cls.pt", "yolov8s-cls.pt", "yolov8m-cls.pt"]

# Hiperparametros de entrenamiento
EPOCHS = 15        # Numero de epocas de entrenamiento
IMG_SIZE = 224     # Tamaño de las imagenes (224x224 es estandar para clasificacion)

# Directorio donde se guardaran los resultados
OUTPUT_DIR = "runs/classify"

### Entrenamiento de los modelos

In [ ]:
# Diccionario para almacenar los resultados de cada modelo
resultados_entrenamiento = {}

for modelo_nombre in modelos:
    print(f"\n{'='*60}")
    print(f"Iniciando entrenamiento para: {modelo_nombre}")
    print(f"{'='*60}")

    # Cargar el modelo pre-entrenado de clasificacion
    model = YOLO(modelo_nombre)

    # Entrenar el modelo
    # - data: ruta a la carpeta que contiene 'train' y 'test'
    # - epochs: numero de pasadas completas por el dataset
    # - imgsz: tamanio al que se redimensionan las imagenes
    # - project: carpeta base para guardar resultados
    # - name: nombre del experimento (subcarpeta)
    results = model.train(
        data=dataset_path,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        project=OUTPUT_DIR,
        name=f"train_{modelo_nombre.replace('.pt', '')}"
    )

    # Guardar referencia a los resultados
    resultados_entrenamiento[modelo_nombre] = results
    
    print(f"\nEntrenamiento de {modelo_nombre} finalizado.")
    print(f"Resultados guardados en: {results.save_dir}")

### Visualizacion de las graficas de entrenamiento
YOLO genera automaticamente graficas durante el entrenamiento. Vamos a cargarlas y mostrarlas para analizar el comportamiento de cada modelo.

In [ ]:
def mostrar_graficas_entrenamiento(output_dir, modelos):
    """
    Muestra las graficas de entrenamiento generadas por YOLO para cada modelo.
    YOLO genera automaticamente:
    - results.png: graficas de loss y metricas por epoca
    - confusion_matrix.png: matriz de confusion del conjunto de validacion
    """
    nombres_modelos = [m.replace('.pt', '') for m in modelos]
    
    for nombre in nombres_modelos:
        # Buscar la carpeta del experimento
        carpeta_exp = os.path.join(output_dir, f"train_{nombre}")
        
        if not os.path.exists(carpeta_exp):
            print(f"No se encontro la carpeta: {carpeta_exp}")
            continue
            
        print(f"\n{'='*60}")
        print(f"Graficas para: {nombre}")
        print(f"{'='*60}")
        
        # Mostrar results.png (curvas de loss y metricas)
        results_img = os.path.join(carpeta_exp, "results.png")
        if os.path.exists(results_img):
            fig, ax = plt.subplots(figsize=(15, 10))
            img = Image.open(results_img)
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(f"Curvas de entrenamiento - {nombre}", fontsize=14)
            plt.tight_layout()
            plt.show()
        else:
            print(f"No se encontro results.png en {carpeta_exp}")
        
        # Mostrar matriz de confusion
        confusion_img = os.path.join(carpeta_exp, "confusion_matrix.png")
        if os.path.exists(confusion_img):
            fig, ax = plt.subplots(figsize=(10, 10))
            img = Image.open(confusion_img)
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(f"Matriz de Confusion - {nombre}", fontsize=14)
            plt.tight_layout()
            plt.show()

# Mostrar las graficas de todos los modelos
mostrar_graficas_entrenamiento(OUTPUT_DIR, modelos)